In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv('Lab2.csv')

X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

 X_out['total_charges'] / (X_out['tenure'] + 1)
        return X_out

In [ ]:
# ==========================================================
# Task 1: Custom Transformers Implementation
# ==========================================================

class TextToNumericCleaner(BaseEstimator, TransformerMixin):
    """ทำความสะอาดคอลัมน์ที่เป็น Text ให้เป็น Numeric Float"""
    def __init__(self, column_name='total_charges'):
        self.column_name = column_name

    def fit(self, X, y=None):
        return self  # ไม่มี state ต้องเรียนรู้

    def transform(self, X):
        X_out = X.copy()
        if self.column_name in X_out.columns:
            # แปลงค่าที่ไม่ใช่ตัวเลข หรือช่องว่างเปล่า ให้เป็น NaN
            X_out[self.column_name] = pd.to_numeric(X_out[self.column_name], errors='coerce')
        return X_out

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """สร้าง Feature ใหม่จากการคำนวณทางธุรกิจ"""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        # ป้องกัน division by zero ด้วย +1
        X_out['charge_per_month'] =

In [ ]:
# ==========================================================
# Task 2: Advanced Preprocessing & Column Mapping
# ==========================================================

# ระบุกลุ่มตัวแปร (รวมคอลัมน์ใหม่ที่จะถูกสร้างขึ้นด้วย)
num_cols = ['tenure', 'monthly_charges', 'total_charges', 'charge_per_month']
cat_cols = ['contract_type', 'payment_method']

# Pipeline สำหรับตัวเลข (ใช้ RobustScaler เพื่อทนทานต่อ Outlier)
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Pipeline สำหรับ Categorical
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer หลัก
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# ==========================================================
# Task 3: Full Pipeline Assembly & GridSearchCV
# ==========================================================

# รวม Custom Transformers เข้ากับ Preprocessor และ Model
full_pipeline = Pipeline(steps=[
    ('text_cleaner', TextToNumericCleaner(column_name='total_charges')),
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('model', RandomForestClassifier(random_state=42))
])

# กำหนด Parameter Grid เพื่อทดสอบผ่าน Grid Search
param_grid = {
    'feature_selection__k': [3, 5],
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 5]
}

# ตั้งค่า 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# สร้าง GridSearchCV
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1', # เน้น F1-score เนื่องจาก Class Imbalance
    n_jobs=-1
)

In [ ]:
# ---------------------------------------------------------
# Execution & Evaluation
# ---------------------------------------------------------
print("กำลังทำการ Cross-Validation และ Hyperparameter Tuning...")
grid_search.fit(X_train, y_train)

print("\n=========================================")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")
print("=========================================\n")

# Evaluation บน Test Set ด้วย Best Estimator ที่ได้
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Set Classification Report:")
print(classification_report(y_test, y_pred))